# Kurir Banjir

| Name | NIM | Email |
|---|---|---|
| Elifele Fredrik Imanuel Jelifta | 0706012410024 | efredrik@student.ciputra.ac.id |
| Kenneth Jonathan Halim | 0706012410009 | kjonathan08@student.ciputra.ac.id |
| Rei Putra Soemanto | 0706012410060 | rsoemanto@student.ciputra.ac.id |

## 3. Configuration

In [ ]:
# Coordinate reference system & study area
CRS = "ESPG:32749"
PLACE_SUFFIX = ", Surabaya, East Java, Indonesia"
KECAMATAN = ["Gubeng", "Gunung Anyar", "Sukolilo", "Tambaksari", "Mulyorejo", "Rungkut", "Tenggilis Mejoyo"]
NETWORK_TYPE = "drive"

# Flood levels
SCENARIOS = {
    "ringan": {"butter_m": 5, "speed_factor": 0.50},
    "sedang": {"butter_m": 15, "speed_factor": 0.25},
    "berat": {"butter_m": 30, "speed_factor": None},
}
LEVEL = ["none", "ringan", "sedang", "berat"]

MIN_INSIDE_M = 20
BASE_HALF_WIDTH = 5
COURIER_MODES = ["pass", "avoid"]

# Depot & destination
DEPOT = {"label": "Depot (ITS)", "query": "Institut Teknologi Sepuluh Nopember, Surabaya, Indonesia", "kecamatan": "Sukolilo"}
DESTINATIONS = [
    {"label": "Ubaya Tenggilis",       "query": "Universitas Surabaya Tenggilis, Surabaya, Indonesia",   "kecamatan": "Rungkut"},
    {"label": "Stasiun Gubeng",        "query": "Stasiun Gubeng, Surabaya, Indonesia",                   "kecamatan": "Tambaksari"},
    {"label": "RSUD Haji",             "query": "Haji Hospital Surabaya",                                "kecamatan": "Sukolilo"},
    {"label": "Kebun Bibit Wonorejo",  "query": "Kebun Bibit Wonorejo, Surabaya",                        "kecamatan": "Rungkut"},
    {"label": "UPN Veteran Jatim",     "query": "Universitas Pembangunan Nasional Veteran Jawa Timur",   "kecamatan": "Gunung Anyar"},
    {"label": "Unair Kampus C",        "query": "Universitas Airlangga Kampus C",                        "kecamatan": "Mulyorejo"},
    {"label": "Pakuwon City Mall",     "query": "Pakuwon City Mall, Surabaya, Indonesia",                "kecamatan": "Mulyorejo"},
    {"label": "Gelora 10 November",    "query": "Gelora 10 November, Surabaya",                          "kecamatan": "Tambaksari"},
]

# Flood zones
ZONE_SPECS = [
    {
        "zone": "Jl. Panjang Jiwo", "kecamatan": "Tenggilis Mejoyo", "name_regex": r"^Jalan Panjang Jiwo$", "extent": (0.0, 1.0),
        "evidence": "[2] worst flooding on 5 Nov 2025 ('paling parah')", "simulated": False, "enabled": True, "level": "berat"
    },
    {
        "zone": "Jl. Raya Pandugo", "kecamatan": "Rungkut", "name_regex": r"^Jalan Raya Pandugo$", "extent": (0.0, 1.0),
        "evidence": "[8] 50 cm on 23 Jun 2026; [9] flood-prone list", "simulated": False, "enabled": True, "level": "berat"
    },
    {
        "zone": "Jl. Barata Jaya (+ side streets)", "kecamatan": "Gubeng", "name_regex": r"^Jalan Barata Jaya\b", "extent": (0.0, 1.0),
        "evidence": "[8] 30 cm on 23 Jun 2026", "simulated": False, "enabled": True, "level": "sedang"
    },
    {
        "zone": "Jl. Nginden Intan Timur I-XIX", "kecamatan": "Sukolilo", "name_regex": r"^Jalan Nginden Intan Timur\b", "extent": (0.0, 1.0),
        "evidence": "[8] 20-30 cm on 23 Jun 2026", "simulated": False, "enabled": True, "level": "sedang"
    },
    {
        "zone": "Jl. Puri Gunung Anyar Regency", "kecamatan": "Gunung Anyar", "name_regex": r"^Jalan Puri Gunung Anyar Regency$", "extent": (0.0, 1.0),
        "evidence": "[8] 30 cm on 23 Jun 2026", "simulated": False, "enabled": True, "level": "sedang"
    },
    {
        "zone": "Jl. Raya Mulyosari (middle 50%)", "kecamatan": "Mulyorejo", "name_regex": r"^Jalan Raya Mulyosari$", "extent": (0.25, 0.75),
        "evidence": "[10] 'langganan genangan' (2024); [9] flood-prone list", "simulated": False, "enabled": True, "level": "ringan"
    },
    {
        "zone": "Jl. Kenjeran (middle 50%)", "kecamatan": "Tambaksari", "name_regex": r"^Jalan Kenjeran$", "extent": (0.25, 0.75),
        "evidence": "[9] flood-prone list only - no flood event reported", "simulated": True, "enabled": True, "level": "ringan"
    },
]

## 4. Setup

In [ ]:
import os
import re
import time
import itertools
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
import folium
import xyzservices.providers as xyz
from branca.colormap import LinearColormap
import shapely
from shapely.geometry import LineString, Point, Polygon, box
from shapely.ops import unary_union
from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

ox.settings.use_cache = True
ox.settings.log_console = False
ox.settings.requests_timeout = 300

OVERPASS_URLS = ["https://maps.mail.ru/osm/tools/overpass/api", "https://overpass-api.de/api", "https://overpass.kumi.systems/api"]
RAW_GRAPH_FILE = "data/east_surabaya_drive_raw.graphml"

def with_retry(fn, *args, attempts=5, **kwargs):
    last = None
    for attempt in range(attempts):
        for url in OVERPASS_URLS:
            ox.settings.overpass_url = url
            ox.settings.overpass_rate_limit = "overpass-api.de" in url
            try:
                return fn(*args, **kwargs)
            except Exception as exc:
                last = exc
                print(f"  {url} failed ({type(exc).__name__}), trying next server")
        time.sleep(20 * 2 ** attempt)
    raise last

print("osmnx", ox.__version__, "| geopandas", gpd.__version__, "| networkx", nx.__version__, "| folium", folium.__version__)

### Interactive map helpers

In [ ]:
MAP_HEIGHT = "650px"
EFFECT_COLOR = {"slowed": "#ff8c00", "closed": "d62728"}
ONE_STYLE = {"color": "#1f4e9c", "weight": 3, "fillColor": "#1f77b4", "fillOpacity": 0.55}

def to_web(gdf, cols):
    out = gdf[cols + ["geometry"]].to_crs(4326)
    out["geometry"] = shapely.set_precision(out.geometry.values, 1e-6)
    return out

def new_map():
    fig = folium.Figure(height=MAP_HEIGHT)
    m = folium.Map(tiles=None, control_scale=True, zoom_snap=0.25, zoom_delta=0.5).add_to(fig)
    folium.TileLayer(xyz.Esri.WorldGrayCanvas, name="Basemap: light grey (Esri)").add_to(m)  # keyless, unlike CARTO
    folium.TileLayer("OpenStreetMap", name="Basemap: OpenStreetMap", show=False).add_to(m)
    return m

def add_geojson(parent, gdf, fields, aliases, style, name=None, show=True):
    folium.GeoJson(
        to_web(gdf, fields), name=name, show=show, control=name is not None,
        style_function=lambda f: style(f["properties"]),
        highlight_function=lambda f: {"weight": 8, "fillOpacity": 0.8},
        tooltip=folium.GeoJsonTooltip(fields=fields, aliases=aliases, sticky=True),
    ).add_to(parent)

def add_boundaries(m):
    add_geojson(m, boundaries, ["kecamatan", "area_km2"], ["Kecamatan", "Area (km²)"],
                lambda p: {"color": "black", "weight": 1.5, "fill": False}, name="Kecamatan boundaries")

def add_zones(parent, zl, name=None, show=True):
    web = zl.assign(centreline_m=zl.centreline_m.round(0), area_m2=zl.area.round(0), osm_names=zl.osm_names.str.join(", "))
    add_geojson(parent, web, ["zone", "kecamatan", "level", "simulated", "centreline_m", "area_m2", "osm_names", "evidence"],
                ["Zone", "Kecamatan", "Level", "Simulated", "Road lines (m)", "Area (m²)", "OSM roads", "Evidence"],
                lambda p: {**ZONE_STYLE, "dashArray": "6 4" if p["simulated"] else None}, name=name, show=show)

def add_points(parent, pts, name="Depot & destinations"):
    fg = folium.FeatureGroup(name=name).add_to(parent)
    for _, r in pts.to_crs(4326).iterrows():
        tip = (f"<b>{r.label}</b> ({r.role})<br>OSM: {r.osm_name}<br>Kecamatan: {r.kecamatan}" f"<br>Snap distance: {r.snap_dist_m:.0f} m")
        if r.role == "depot":
            folium.Marker([r.geometry.y, r.geometry.x], tooltip=tip, icon=folium.Icon(color="orange", icon="star")).add_to(fg)
        else:
            folium.CircleMarker([r.geometry.y, r.geometry.x], radius=7, color="white", weight=2, fill=True, fill_color="#1f77b4", fill_opacity=1, tooltip=tip).add_to(fg)


def add_legend(m, title, items):
    rows = "".join(f'<div><span style="display:inline-block;width:22px;height:10px;margin-right:6px;' f'vertical-align:middle;{css}"></span>{label}</div>' for label, css in items)
    m.get_root().html.add_child(folium.Element(
        f'<div style="position:absolute;bottom:28px;left:10px;z-index:9999;background:white;padding:6px 9px;'
        f'border:1px solid #999;border-radius:4px;font:12px sans-serif"><b>{title}</b>{rows}</div>'))


def show_map(m, extent_gdf, pad_m=500):
    w, s, e, n = gpd.GeoSeries([box(*extent_gdf.total_bounds).buffer(pad_m)], crs=CRS).to_crs(4326).total_bounds
    m.fit_bounds([[s, w], [n, e]])
    folium.LayerControl(collapsed=False).add_to(m)
    return m.get_root()